<a href="https://colab.research.google.com/github/Ritvik2090/loan-portfolio-analytics/blob/main/notebooks/01_payment_event_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading data and EMI sorting and grouping

In [ ]:
import pandas as pd
import numpy as np
import random

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

random.seed(42)
np.random.seed(42)

In [57]:
from google.colab import files

uploaded = files.upload()

Mounted at /content/drive


In [ ]:
customers = pd.read_csv("customers.csv")
loan_products = pd.read_csv("loan_products.csv")
loans = pd.read_csv("loans.csv")
emis = pd.read_csv("emis.csv")

In [ ]:
# Loans
loan_date_columns = [
    "disbursed_date",
    "closure_date",
    "created_at",
    "updated_at"
]

for col in loan_date_columns:
    loans[col] = pd.to_datetime(loans[col])


# EMIs
emi_date_columns = [
    "due_date",
    "created_at",
    "updated_at"
]

for col in emi_date_columns:
    emis[col] = pd.to_datetime(emis[col])

In [ ]:
print(loans.dtypes)
print()
print(emis.dtypes)

id                         int64
customer_id                int64
product_id                 int64
loan_amount                int64
tenure_months              int64
disbursed_date    datetime64[ns]
loan_status               object
closure_date      datetime64[ns]
created_at        datetime64[ns]
updated_at        datetime64[ns]
dtype: object

id                           int64
loan_id                      int64
due_date            datetime64[ns]
emi_amount                   int64
principal_amount             int64
interest_amount              int64
status                      object
created_at          datetime64[ns]
updated_at          datetime64[ns]
dtype: object


In [ ]:
print(customers.shape)
print(loan_products.shape)
print(loans.shape)
print(emis.shape)

(10000, 13)
(8, 11)
(15000, 10)
(199725, 9)


In [ ]:
print(loans.isnull().sum())

print()

print(emis.isnull().sum())

id                    0
customer_id           0
product_id            0
loan_amount           0
tenure_months         0
disbursed_date        0
loan_status           0
closure_date      10491
created_at            0
updated_at            0
dtype: int64

id                  0
loan_id             0
due_date            0
emi_amount          0
principal_amount    0
interest_amount     0
status              0
created_at          0
updated_at          0
dtype: int64


In [ ]:
emis = emis.sort_values(
    by=["loan_id", "due_date", "id"]
).reset_index(drop=True)

In [ ]:
profiles = [
    "Excellent",
    "Regular",
    "Late",
    "Partial",
    "Delinquent",
    "Foreclosure"
]

weights = [
    35,
    35,
    15,
    8,
    5,
    2
]

In [ ]:
loan_profiles = {}

for loan_id in loans["id"]:

    loan_profiles[loan_id] = random.choices(
        profiles,
        weights=weights,
        k=1
    )[0]

In [ ]:
from collections import Counter

profile_distribution = Counter(
    loan_profiles.values()
)

profile_distribution

Counter({'Regular': 5246,
         'Excellent': 5257,
         'Late': 2278,
         'Partial': 1147,
         'Delinquent': 761,
         'Foreclosure': 311})

# Generating payments and events data

In [ ]:
payment_events = []

payment_event_id = 1

In [ ]:
payment_modes = [
    "eMandate",
    "UPI",
    "Net Banking",
    "Debit Card",
    "Bank Transfer"
]

In [ ]:
for loan in loans.itertuples(index=False):

    loan_id = loan.id

    profile = loan_profiles[loan_id]

    loan_emis = emis[
        emis["loan_id"] == loan_id
    ].copy()

## Excellent Payment Events

In [ ]:
def generate_excellent_payment_events(loan_emis):

    events = []

    for emi in loan_emis.itertuples(index=False):

        # Excellent borrowers mostly use eMandate
        payment_mode = random.choices(
            payment_modes,
            weights=[90, 5, 2, 2, 1],
            k=1
        )[0]

        # Business Rule:
        # eMandate always debits on the due date.
        if payment_mode == "eMandate":
            payment_date = emi.due_date

        # Customer-initiated payments may be 0-2 days early.
        else:
            payment_date = emi.due_date - timedelta(
                days=random.randint(0, 2)
            )

        events.append({
            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode
        })

    return events

In [ ]:
sample_loan = loans.iloc[0]

sample_emis = emis[
    emis["loan_id"] == sample_loan["id"]
]

events = generate_excellent_payment_events(sample_emis)

events[:5]

[{'payment_date': Timestamp('2025-09-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2025-10-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Bank Transfer'},
 {'payment_date': Timestamp('2025-11-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2025-12-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2026-01-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'}]

## Regular Payment Events

In [ ]:
def generate_regular_payment_events(loan_emis):

    events = []

    for emi in loan_emis.itertuples(index=False):

        payment_mode = random.choices(
            payment_modes,
            weights=[70, 20, 4, 3, 3],
            k=1
        )[0]

        # eMandate always debits on due date
        if payment_mode == "eMandate":

            payment_date = emi.due_date

        else:

            timing = random.choices(
                ["early", "ontime", "late"],
                weights=[15, 70, 15],
                k=1
            )[0]

            if timing == "early":

                payment_date = emi.due_date - timedelta(
                    days=random.randint(1, 3)
                )

            elif timing == "late":

                payment_date = emi.due_date + timedelta(
                    days=random.randint(1, 3)
                )

            else:

                payment_date = emi.due_date

        events.append({
            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode
        })

    return events

In [ ]:
events = generate_regular_payment_events(sample_emis)

events[:10]

[{'payment_date': Timestamp('2025-09-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2025-10-23 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Debit Card'},
 {'payment_date': Timestamp('2025-11-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2025-12-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Bank Transfer'},
 {'payment_date': Timestamp('2026-01-29 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2026-02-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2026-03-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'}]

## Late Payment Events

In [ ]:
def generate_late_payment_events(loan_emis):

    events = []

    manual_payment_modes = [
        "UPI",
        "Net Banking",
        "Debit Card",
        "Bank Transfer"
    ]

    for emi in loan_emis.itertuples(index=False):

        payment_mode = random.choices(
            manual_payment_modes,
            weights=[60, 20, 10, 10],
            k=1
        )[0]

        payment_date = emi.due_date + timedelta(
            days=random.randint(5, 15)
        )

        events.append({
            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode
        })

    return events

In [ ]:
events = generate_late_payment_events(sample_emis)

events[:10]

[{'payment_date': Timestamp('2025-10-10 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Net Banking'},
 {'payment_date': Timestamp('2025-10-31 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2025-12-03 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2026-01-05 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2026-02-02 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2026-03-07 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'},
 {'payment_date': Timestamp('2026-04-08 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'UPI'}]

## Partial Payment Events

In [71]:
def split_amount(total_amount, n_parts):

    # Decide a safe minimum percentage automatically
    max_percent = min(
        0.30,
        (1 / n_parts) - 0.02
    )

    min_percent = random.uniform(
        0.10,
        max_percent
    )

    minimum = int(total_amount * min_percent)

    # Ensure the split is possible
    if minimum * n_parts > total_amount:
        raise ValueError(
            f"Cannot split ₹{total_amount} into {n_parts} parts "
            f"with minimum {min_percent:.2%} each."
        )

    # Give every payment the minimum amount
    amounts = np.full(n_parts, minimum, dtype=int)

    remaining = total_amount - amounts.sum()

    # If nothing remains, return
    if remaining == 0:
        return amounts.tolist()

    # Randomly distribute the remaining amount
    weights = np.random.random(n_parts)
    weights /= weights.sum()

    extra = np.floor(weights * remaining).astype(int)

    # Handle rounding difference
    difference = remaining - extra.sum()

    for i in range(difference):
        extra[i % n_parts] += 1

    amounts += extra

    # Shuffle so the smallest amount isn't always first
    np.random.shuffle(amounts)

    return amounts.tolist()

In [72]:
print(split_amount(10000, 2))
print(split_amount(10000, 3))
print(split_amount(10000, 4))

[3555, 6445]
[2679, 4114, 3207]
[2824, 2267, 2709, 2200]


In [73]:
for parts in [2, 3, 4]:

    values = split_amount(10000, parts)

    print(values)

    print("Sum :", sum(values))
    print("Parts:", len(values))
    print("Positive:", all(x > 0 for x in values))
    print("-" * 40)

[3638, 6362]
Sum : 10000
Parts: 2
Positive: True
----------------------------------------
[3007, 2042, 4951]
Sum : 10000
Parts: 3
Positive: True
----------------------------------------
[2382, 2506, 2585, 2527]
Sum : 10000
Parts: 4
Positive: True
----------------------------------------


In [81]:
def generate_partial_payment_events(loan_emis):

    events = []

    manual_payment_modes = [
        "UPI",
        "Net Banking",
        "Debit Card",
        "Bank Transfer"
    ]

    for emi in loan_emis.itertuples(index=False):

        # ---------------------------------
        # Decide whether this EMI is split
        # ---------------------------------

        split = random.random() < 0.15

        # =================================
        # NORMAL EMI
        # =================================

        if not split:

            payment_mode = random.choices(
                payment_modes,
                weights=[70, 20, 4, 3, 3],
                k=1
            )[0]

            if payment_mode == "eMandate":

                payment_date = emi.due_date

            else:

                timing = random.choices(
                    ["early", "ontime", "late"],
                    weights=[15, 70, 15],
                    k=1
                )[0]

                if timing == "early":

                    payment_date = emi.due_date - timedelta(
                        days=random.randint(1, 3)
                    )

                elif timing == "late":

                    payment_date = emi.due_date + timedelta(
                        days=random.randint(1, 3)
                    )

                else:

                    payment_date = emi.due_date

            events.append({

                "payment_date": payment_date,
                "paid_amount": emi.emi_amount,
                "payment_mode": payment_mode

            })

        # =================================
        # PARTIAL EMI
        # =================================

        else:

            n_events = random.choices(
                [2, 3, 4],
                weights=[80, 15, 5],
                k=1
            )[0]


            # Partial borrowers usually make meaningful payments.
            # Every payment event should be at least 15-30% of the EMI.
            amounts = split_amount(
                total_amount = emi.emi_amount,
                n_parts = n_events
            )

            # Small probability that first payment
            # is through eMandate
            use_emandate = random.random() < 0.10

            current_date = emi.due_date

            for i in range(n_events):

                if i == 0 and use_emandate:

                    payment_mode = "eMandate"
                    payment_date = emi.due_date

                else:

                    payment_mode = random.choices(
                        manual_payment_modes,
                        weights=[60, 20, 10, 10],
                        k=1
                    )[0]

                    if i == 0:

                        payment_date = emi.due_date

                    else:

                        current_date += timedelta(
                            days=random.randint(1, 5)
                        )

                        payment_date = current_date

                events.append({

                    "payment_date": payment_date,
                    "paid_amount": amounts[i],
                    "payment_mode": payment_mode

                })

    return events

In [75]:
events = generate_partial_payment_events(sample_emis)

events[:20]

[{'payment_date': Timestamp('2025-09-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2025-10-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Bank Transfer'},
 {'payment_date': Timestamp('2025-11-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2025-12-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2026-01-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'Bank Transfer'},
 {'payment_date': Timestamp('2026-02-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'},
 {'payment_date': Timestamp('2026-03-26 00:00:00'),
  'paid_amount': 3418,
  'payment_mode': 'eMandate'}]

## Foreclosure Payment Events

In [76]:
def generate_foreclosure_payment_events(loan_emis):


# ------------------------------------
# Foreclosure
# One payment event settles all
# remaining EMIs from the foreclosure
# EMI onwards.
# ------------------------------------

    events = []

    manual_payment_modes = [
        "UPI",
        "Net Banking",
        "Debit Card",
        "Bank Transfer"
    ]

    tenure = len(loan_emis)

    start_emi = min(6, tenure)
    end_emi = min(12, tenure)

    foreclosure_emi = random.randint(
        start_emi,
        end_emi
    )

    # ------------------------------------
    # Generate normal payment events
    # before foreclosure
    # ------------------------------------

    for emi in loan_emis.iloc[:foreclosure_emi - 1].itertuples(index=False):

        payment_mode = random.choices(
            payment_modes,
            weights=[70, 20, 4, 3, 3],
            k=1
        )[0]

        if payment_mode == "eMandate":

            payment_date = emi.due_date

        else:

            timing = random.choices(
                ["early", "ontime", "late"],
                weights=[15, 70, 15],
                k=1
            )[0]

            if timing == "early":

                payment_date = (
                    emi.due_date
                    - timedelta(days=random.randint(1, 3))
                )

            elif timing == "late":

                payment_date = (
                    emi.due_date
                    + timedelta(days=random.randint(1, 3))
                )

            else:

                payment_date = emi.due_date

        events.append({

            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode

        })

    # ------------------------------------
    # Foreclosure payment
    # ------------------------------------

    remaining_emis = loan_emis.iloc[foreclosure_emi - 1:]

    remaining_balance = int(
        remaining_emis["emi_amount"].sum()
    )

    foreclosure_date = remaining_emis.iloc[0]["due_date"]

    payment_mode = random.choices(
        manual_payment_modes,
        weights=[60, 20, 10, 10],
        k=1
    )[0]

    events.append({

        "payment_date": foreclosure_date,
        "paid_amount": remaining_balance,
        "payment_mode": payment_mode

    })

    return events

In [66]:
events = generate_foreclosure_payment_events(sample_emis)

for event in events:
    print(event)

{'payment_date': Timestamp('2025-09-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'UPI'}
{'payment_date': Timestamp('2025-10-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}
{'payment_date': Timestamp('2025-11-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'UPI'}
{'payment_date': Timestamp('2025-12-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}
{'payment_date': Timestamp('2026-01-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}
{'payment_date': Timestamp('2026-02-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}
{'payment_date': Timestamp('2026-03-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'UPI'}


## Deliquent Payment Events

In [77]:
def generate_delinquent_payment_events(loan_emis):

    events = []

    manual_payment_modes = [
        "UPI",
        "Net Banking",
        "Debit Card",
        "Bank Transfer"
    ]

    tenure = len(loan_emis)

    # Delinquency should not happen too early or too late
    miss_start = random.randint(
        3,
        max(3, tenure - 3)
    )

    # Miss either 1 or 2 EMIs
    miss_count = random.choice([1, 2])

    miss_end = min(
        miss_start + miss_count - 1,
        tenure - 1
    )

    # ------------------------------------
    # Normal payments before delinquency
    # ------------------------------------

    for emi in loan_emis.iloc[:miss_start - 1].itertuples(index=False):

        payment_mode = random.choices(
            payment_modes,
            weights=[70, 20, 4, 3, 3],
            k=1
        )[0]

        if payment_mode == "eMandate":

            payment_date = emi.due_date

        else:

            timing = random.choices(
                ["early", "ontime", "late"],
                weights=[15, 70, 15],
                k=1
            )[0]

            if timing == "early":

                payment_date = emi.due_date - timedelta(
                    days=random.randint(1, 3)
                )

            elif timing == "late":

                payment_date = emi.due_date + timedelta(
                    days=random.randint(2, 7)
                )

            else:

                payment_date = emi.due_date

        events.append({

            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode

        })

    # ------------------------------------
    # Catch-up payment
    # (Missed EMIs + Current EMI)
    # ------------------------------------

    current_emi = miss_end + 1

    catchup_emis = loan_emis.iloc[
        miss_start - 1 : current_emi
    ]

    catchup_amount = int(
        catchup_emis["emi_amount"].sum()
    )

    catchup_date = (
        loan_emis.iloc[current_emi - 1].due_date
        + timedelta(days=random.randint(2, 7))
    )

    payment_mode = random.choices(
        manual_payment_modes,
        weights=[60, 20, 10, 10],
        k=1
    )[0]

    events.append({

        "payment_date": catchup_date,
        "paid_amount": catchup_amount,
        "payment_mode": payment_mode

    })

    # ------------------------------------
    # Resume normal payments
    # ------------------------------------

    for emi in loan_emis.iloc[current_emi:].itertuples(index=False):

        payment_mode = random.choices(
            payment_modes,
            weights=[70, 20, 4, 3, 3],
            k=1
        )[0]

        if payment_mode == "eMandate":

            payment_date = emi.due_date

        else:

            timing = random.choices(
                ["early", "ontime", "late"],
                weights=[15, 70, 15],
                k=1
            )[0]

            if timing == "early":

                payment_date = emi.due_date - timedelta(
                    days=random.randint(1, 3)
                )

            elif timing == "late":

                payment_date = emi.due_date + timedelta(
                    days=random.randint(1, 3)
                )

            else:

                payment_date = emi.due_date

        events.append({

            "payment_date": payment_date,
            "paid_amount": emi.emi_amount,
            "payment_mode": payment_mode

        })

    return events

In [68]:
events = generate_delinquent_payment_events(sample_emis)

for event in events:
    print(event)

{'payment_date': Timestamp('2025-09-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}
{'payment_date': Timestamp('2025-10-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'UPI'}
{'payment_date': Timestamp('2026-01-31 00:00:00'), 'paid_amount': 10254, 'payment_mode': 'Net Banking'}
{'payment_date': Timestamp('2026-02-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'UPI'}
{'payment_date': Timestamp('2026-03-26 00:00:00'), 'paid_amount': 3418, 'payment_mode': 'eMandate'}


# Verification

In [82]:
events = generate_delinquent_payment_events(sample_emis)

total_paid = sum(
    event["paid_amount"]
    for event in events
)

total_emi = int(
    sample_emis["emi_amount"].sum()
)

print("Paid :", total_paid)
print("EMIs :", total_emi)
print("PASS :", total_paid == total_emi)

Paid : 23926
EMIs : 23926
PASS : True


In [83]:
events = generate_partial_payment_events(sample_emis)

for event in events:

    assert set(event.keys()) == {
        "payment_date",
        "paid_amount",
        "payment_mode"
    }

print("PASS")

PASS


In [84]:
events = generate_partial_payment_events(sample_emis)

payment_modes_found = {
    event["payment_mode"]
    for event in events
}

print(payment_modes_found)

{'UPI', 'eMandate'}


In [85]:
events = generate_regular_payment_events(sample_emis)

for event in events:

    if event["payment_mode"] == "eMandate":

        assert event["payment_date"] in set(
            sample_emis["due_date"]
        )

print("PASS")

PASS


In [86]:
events = generate_partial_payment_events(sample_emis)

print("Payment Events :", len(events))
print("EMIs :", len(sample_emis))

Payment Events : 11
EMIs : 7


In [87]:
events = generate_foreclosure_payment_events(sample_emis)

print("Payment Events :", len(events))
print("EMIs :", len(sample_emis))

Payment Events : 6
EMIs : 7


In [88]:
events = generate_delinquent_payment_events(sample_emis)

print("Payment Events :", len(events))
print("EMIs :", len(sample_emis))

Payment Events : 6
EMIs : 7


In [89]:
events = generate_foreclosure_payment_events(sample_emis)

assert events[-1]["payment_mode"] != "eMandate"

print("PASS")

PASS


In [90]:
events = generate_delinquent_payment_events(sample_emis)

emi_amount = sample_emis.iloc[0]["emi_amount"]

catchup_events = [
    e
    for e in events
    if e["paid_amount"] > emi_amount
]

assert len(catchup_events) == 1

manual_modes = {
    "UPI",
    "Debit Card",
    "Net Banking",
    "Bank Transfer"
}

assert catchup_events[0]["payment_mode"] in manual_modes

print("PASS")

PASS


In [91]:
events = generate_delinquent_payment_events(sample_emis)

for i, event in enumerate(events, start=1):

    print(
        f"{i:02d}",
        event["payment_date"].date(),
        event["paid_amount"],
        event["payment_mode"]
    )

01 2025-09-26 3418 eMandate
02 2025-10-26 3418 eMandate
03 2025-11-26 3418 eMandate
04 2026-01-28 6836 Debit Card
05 2026-02-26 3418 eMandate
06 2026-03-26 3418 eMandate


# Payment Events Table

In [92]:
profile_weights = {
    "excellent": 0.40,
    "regular": 0.35,
    "late": 0.10,
    "partial": 0.08,
    "delinquent": 0.05,
    "foreclosure": 0.02,
}

profiles = list(profile_weights.keys())
weights = list(profile_weights.values())

In [93]:
payment_events_rows = []
payment_event_id = 1

for loan_id, loan_emis in emis.groupby("loan_id", sort=False):

    profile = random.choices(
        profiles,
        weights=weights,
        k=1
    )[0]

    if profile == "excellent":
        events = generate_excellent_payment_events(loan_emis)

    elif profile == "regular":
        events = generate_regular_payment_events(loan_emis)

    elif profile == "late":
        events = generate_late_payment_events(loan_emis)

    elif profile == "partial":
        events = generate_partial_payment_events(loan_emis)

    elif profile == "delinquent":
        events = generate_delinquent_payment_events(loan_emis)

    else:
        events = generate_foreclosure_payment_events(loan_emis)

    for event in events:

        payment_events_rows.append({
            "id": payment_event_id,
            "loan_id": loan_id,
            "paid_amount": int(event["paid_amount"]),
            "payment_date": pd.to_datetime(event["payment_date"]).date(),
            "payment_mode": event["payment_mode"],
            "created_at": datetime.today().date(),
            "updated_at": datetime.today().date(),
        })

        payment_event_id += 1

In [94]:
payment_events = pd.DataFrame(payment_events_rows)

payment_events = payment_events.sort_values(
    ["loan_id", "payment_date", "id"]
).reset_index(drop=True)

payment_events.head()

,id,loan_id,paid_amount,payment_date,payment_mode,created_at,updated_at
0,1,1,3418,2025-09-26,Net Banking,2026-07-21,2026-07-21
1,2,1,3418,2025-10-26,eMandate,2026-07-21,2026-07-21
2,3,1,3418,2025-11-26,eMandate,2026-07-21,2026-07-21
3,4,1,3418,2025-12-25,UPI,2026-07-21,2026-07-21
4,5,1,3418,2026-01-26,eMandate,2026-07-21,2026-07-21


In [95]:
payment_events.to_csv(
    "payment_events.csv",
    index=False
)

print("Saved payment_events.csv with", len(payment_events), "rows")

Saved payment_events.csv with 200072 rows
